# Multimodal Financial Market Analysis — BBCA
### Two-Model NeuralProphet: Baseline vs Hybrid (Monthly Financial Reports)

**Authors**: Wesley Coa, Geoffrey Gohtama, Wilbert — Bina Nusantara University, 2026

---

## What changed from the previous version

| Item | Old | New |
|---|---|---|
| Fundamental source | `BBCA_fundamental_extended.json` (44 quarters, 31 *estimated_seasonal*) | Monthly OJK PDFs — actual published figures |
| Granularity | Quarterly (~44 points) | Monthly (~120 points) |
| Data quality | Mostly synthetic seasonal split | 100% official / derived from official |
| NLP / FinBERT | FinBERT on placeholder MD&A text | **Removed** — monthly reports are structured tables, not prose |
| Growth features | QoQ | MoM + YoY |
| New features | — | `nim_proxy`, `cost_income_ratio`, `loans_to_assets` |

### Why no BERT?
BCA's monthly publication (*Laporan Keuangan Publikasi Bulanan*) is an OJK-mandated structured
financial statement — balance sheet, P&L, and commitments in a fixed table layout.
There is **no narrative text** to classify. BERT/FinBERT converts *prose* into numbers;
here the numbers are already there. Using BERT would add computational cost with zero
informational gain and risk introducing noise (as confirmed by the prior run where
Hybrid-NLP was 150–335% worse than Baseline on MAE).

If you later add annual-report MD&A sections as a separate text source, FinBERT can
be re-introduced on top of this pipeline.

---

**Required files**:
- `BBCA_2015_2025_Combined.csv` — daily price history (unchanged)
- `monthly_reports/` — folder containing one PDF per month, named `BBCA_YYYY_MM.pdf`
  (e.g. `BBCA_2015_08.pdf` is the August 2015 file you already have)

**Install once**:
```
pip install neuralprophet pdfplumber scikit-learn matplotlib
```

## Imports, Configuration & Setup

In [16]:
"""
================================================================================
  Multimodal Financial Market Analysis — BBCA Stock
  TWO-MODEL COMPARISON
    1. Baseline  — NeuralProphet on Close price only
    2. Hybrid    — Price + monthly financial-report features
================================================================================
  AUTHORS : Wesley Coa, Geoffrey Gohtama, Wilbert
  SCHOOL  : Universitas Bina Nusantara, 2026
================================================================================
  FILES REQUIRED (same folder as this notebook):
    BBCA_2015_2025_Combined.csv   — daily price history
    monthly_reports/              — folder of OJK monthly PDFs
        BBCA_YYYY_MM.pdf          — naming convention (e.g. BBCA_2015_08.pdf)

  PDF PARSING STRATEGY
  ────────────────────
  BCA's monthly publications follow a fixed OJK layout.  pdfplumber extracts
  the raw text; regex patterns then pull the six key line items:
    • Interest income  (Pendapatan Bunga / Interest income)
    • Net profit       (NET PROFIT (LOSS) / Laba Bersih)
    • Total assets     (TOTAL ASSETS / TOTAL ASET)
    • Total loans      (Loans — line 9)
    • Total deposits   (sum of current + savings + time deposit)
    • Personnel exp.   (Personnel expenses — proxy for operating cost)

  Derived features:
    net_profit_margin  = net_profit / interest_income
    nim_proxy          = (interest_income - interest_expense) / total_assets  [if avail.]
    loans_to_assets    = total_loans / total_assets
    cost_income_ratio  = personnel_exp / (interest_income + other_op_income)  [if avail.]
    ni_growth_mom      = net_profit.pct_change(1)
    ni_growth_yoy      = net_profit.pct_change(12)
    rev_growth_mom     = interest_income.pct_change(1)
    rev_growth_yoy     = interest_income.pct_change(12)
================================================================================
"""

import os, re, glob, warnings
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot   as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import mean_absolute_error, mean_squared_error

from neuralprophet import NeuralProphet, set_log_level
set_log_level("ERROR")
try:
    from neuralprophet import set_plotting_backend
    set_plotting_backend("matplotlib")
except ImportError:
    pass

try:
    import pdfplumber
except ImportError:
    raise ImportError("Run: pip install pdfplumber")


# ==============================================================================
#  CONFIGURATION
# ==============================================================================

PRICE_PATH      = "BBCA_2015_2025_Combined.csv"
PDF_DIR         = "monthly_reports"          # folder containing BBCA_YYYY_MM.pdf
PARSED_CACHE    = "BBCA_monthly_parsed.csv"  # cache so PDFs are only parsed once

TEST_SIZE       = 240      # ~1 year held-out test set (trading days)
EPOCHS          = 100
LEARNING_RATE   = 0.005
N_LAGS          = 14
N_FORECASTS     = 1

# Regressor columns built from monthly PDF data
FUNDAMENTAL_COLS = [
    "net_income",
    "revenue",
    "net_profit_margin",
    "loans_to_assets",
    "ni_growth_mom",
    "ni_growth_yoy",
    "rev_growth_mom",
    "rev_growth_yoy",
]

## Section 1 — Load Price Data

In [17]:
# ==============================================================================
#  SECTION 1 — LOAD PRICE DATA  (unchanged from previous version)
# ==============================================================================

def load_price_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    for fmt in ("%m/%d/%Y", "%Y-%m-%d", "%d/%m/%Y"):
        try:
            df["Date"] = pd.to_datetime(df["Date"], format=fmt)
            break
        except (ValueError, TypeError):
            continue
    else:
        df["Date"] = pd.to_datetime(df["Date"], infer_datetime_format=True)

    df = (df.sort_values("Date")
            .reset_index(drop=True)
            .rename(columns={"Date": "ds", "Close": "y"})[["ds", "y"]])

    print(f"[Price]  {df['ds'].min().date()} -> {df['ds'].max().date()}"
          f"  |  {len(df):,} trading days")
    return df

## Section 2 — Parse Monthly OJK PDFs

This section replaces the old `load_fundamental_data()` that read from
`BBCA_fundamental_extended.json`.  

BCA's monthly OJK publication uses a **fixed table layout** that is
consistent across all months/years. The approach:

1. `pdfplumber` extracts the raw text of each PDF.
2. Regex patterns match the exact label strings used by OJK (both English
   and Indonesian variants, since early reports may differ).
3. Extracted values are stored to `BBCA_monthly_parsed.csv` so the PDFs
   only need to be parsed once.
4. Derived ratio features are computed from the raw line items.

### PDF naming convention
Place all PDFs in the `monthly_reports/` folder and name them:
```
BBCA_YYYY_MM.pdf
```
For example, the August 2015 file you already have should be saved as
`monthly_reports/BBCA_2015_08.pdf`.

In [18]:
# ==============================================================================
#  SECTION 2 — PARSE MONTHLY OJK PDFs
# ==============================================================================

# ---------------------------------------------------------------------------
#  Line-item extraction patterns
#  Each tuple: (column_name, list_of_regex_patterns)
#  Patterns match the label text printed in the OJK standard form.
#  The first captured group must be the numeric value (digits, commas, dots).
# ---------------------------------------------------------------------------
_PATTERNS = [
    # ── Income Statement ──────────────────────────────────────────────────
    (
        "interest_income",
        [
            # English layout (post-2015 bilingual)
            r"1\.\s*Interest income\b[\s\S]{0,80}?([\d,\.]{4,})",
            # Indonesian layout
            r"1\.\s*Pendapatan Bunga\b[\s\S]{0,80}?([\d,\.]{4,})",
            # Fallback: any line starting 'Interest income'
            r"Interest income[^\n]{0,60}?([\d,\.]{6,})",
        ],
    ),
    (
        "interest_expense",
        [
            r"2\.\s*Interest expenses[^\n]{0,60}?([\d,\.]{4,})",
            r"2\.\s*Beban Bunga[^\n]{0,60}?([\d,\.]{4,})",
        ],
    ),
    (
        "net_profit",
        [
            r"NET PROFIT \(LOSS\)[^\n]{0,60}?([\d,\.]{4,})",
            r"LABA \(RUGI\) BERSIH[^\n]{0,60}?([\d,\.]{4,})",
            r"NET PROFIT[^\n]{0,60}?([\d,\.]{6,})",
            r"Laba Bersih[^\n]{0,60}?([\d,\.]{6,})",
        ],
    ),
    (
        "personnel_expense",
        [
            r"Personnel expenses[^\n]{0,60}?([\d,\.]{4,})",
            r"Beban Tenaga Kerja[^\n]{0,60}?([\d,\.]{4,})",
            r"j\.\s*Personnel expenses[^\n]{0,60}?([\d,\.]{4,})",
        ],
    ),
    # ── Balance Sheet ─────────────────────────────────────────────────────
    (
        "total_assets",
        [
            r"TOTAL ASSETS[^\n]{0,60}?([\d,\.]{6,})",
            r"TOTAL ASET[^\n]{0,60}?([\d,\.]{6,})",
        ],
    ),
    (
        "total_loans",
        [
            # Line 9 in the balance sheet: loans / kredit
            r"9\.\s*Loans[^\n]{0,60}?([\d,\.]{6,})",
            r"9\.\s*Kredit[^\n]{0,60}?([\d,\.]{6,})",
        ],
    ),
    (
        "current_accounts",
        [
            r"1\.\s*Current account[^\n]{0,60}?([\d,\.]{6,})",
            r"1\.\s*Giro[^\n]{0,60}?([\d,\.]{6,})",
        ],
    ),
    (
        "savings_accounts",
        [
            r"2\.\s*Saving account[^\n]{0,60}?([\d,\.]{6,})",
            r"2\.\s*Tabungan[^\n]{0,60}?([\d,\.]{6,})",
        ],
    ),
    (
        "time_deposits",
        [
            r"3\.\s*Time deposit[^\n]{0,60}?([\d,\.]{6,})",
            r"3\.\s*Deposito[^\n]{0,60}?([\d,\.]{6,})",
        ],
    ),
]


def _clean_number(raw: str) -> float:
    """
    Convert an extracted string such as '28,772,308' or '28.772.308'
    to a float.  OJK reports use commas as thousands separator (EN layout)
    or periods (ID layout).  We detect which by position of the last
    separator character.
    """
    raw = raw.strip()
    if not raw:
        return np.nan
    # If the string ends with two digits after the last separator it is
    # a decimal; otherwise it is a thousands separator.
    if re.search(r'[,\.](\d{1,2})$', raw):
        # likely decimal separator at the end — remove thousands sep first
        raw = re.sub(r'[,\.](?=\d{3})', '', raw)
        raw = raw.replace(',', '.').replace(' ', '')
    else:
        raw = re.sub(r'[,\. ]', '', raw)
    try:
        return float(raw)
    except ValueError:
        return np.nan


def _extract_from_text(text: str) -> dict:
    """
    Apply all patterns to the full PDF text and return a dict of
    {column_name: value_in_millions_IDR}.
    Returns NaN for any field not found.
    """
    result = {}
    for col, patterns in _PATTERNS:
        found = np.nan
        for pat in patterns:
            m = re.search(pat, text, re.IGNORECASE)
            if m:
                found = _clean_number(m.group(1))
                if not np.isnan(found):
                    break
        result[col] = found
    return result


def _parse_single_pdf(pdf_path: str) -> dict | None:
    """
    Extract all target line items from one monthly PDF.
    Returns a flat dict {column: value} or None on failure.

    Strategy
    --------
    1. Extract raw text from every page and concatenate.
    2. Apply regex patterns defined in _PATTERNS.
    3. If interest_income or net_profit is missing, flag the file for
       manual review (rare; only occurs if the PDF is scanned image).

    Note on units:
    The OJK monthly report is in millions of Rupiah (IDR millions).
    All values are stored as-is; downstream normalisation converts to
    IDR billions for consistency with price data scale.
    """
    try:
        with pdfplumber.open(pdf_path) as pdf:
            full_text = "\n".join(
                page.extract_text() or "" for page in pdf.pages
            )
    except Exception as e:
        print(f"  [WARN] Could not open {pdf_path}: {e}")
        return None

    values = _extract_from_text(full_text)

    # Sanity check — must have at least interest_income and net_profit
    if np.isnan(values.get("interest_income", np.nan)) or \
       np.isnan(values.get("net_profit",      np.nan)):
        print(f"  [WARN] Key fields missing in {os.path.basename(pdf_path)} "
              f"— check regex or if PDF is image-based")

    return values


def load_monthly_pdf_data(
    pdf_dir   : str,
    cache_path: str,
) -> pd.DataFrame:
    """
    Main entry point for monthly PDF fundamentals.

    Parameters
    ----------
    pdf_dir    : path to folder containing BBCA_YYYY_MM.pdf files
    cache_path : path to CSV cache; PDFs are only re-parsed if missing
                 from cache

    Returns
    -------
    DataFrame with columns:
        ds, net_income, revenue, net_profit_margin,
        loans_to_assets, ni_growth_mom, ni_growth_yoy,
        rev_growth_mom, rev_growth_yoy
    """
    # ── 1. Load cache if it exists ──────────────────────────────────────
    if os.path.exists(cache_path):
        cached = pd.read_csv(cache_path, parse_dates=["ds"])
        cached_months = set(cached["ds"].dt.to_period("M").astype(str))
        print(f"[Monthly]  Cache loaded: {len(cached)} months already parsed.")
    else:
        cached = pd.DataFrame()
        cached_months = set()

    # ── 2. Find PDFs not yet in cache ───────────────────────────────────
    pdf_files = sorted(glob.glob(os.path.join(pdf_dir, "BBCA_????_??.pdf")))
    if not pdf_files:
        raise FileNotFoundError(
            f"No PDFs found in '{pdf_dir}'. "
            "Expected files named BBCA_YYYY_MM.pdf"
        )

    new_rows = []
    for pdf_path in pdf_files:
        fname = os.path.basename(pdf_path)   # e.g. BBCA_2015_08.pdf
        m     = re.match(r"BBCA_(\d{4})_(\d{2})\.pdf", fname)
        if not m:
            continue
        year, month = int(m.group(1)), int(m.group(2))
        # Period end = last day of the reported month
        period_end  = pd.Timestamp(year, month, 1) + pd.offsets.MonthEnd(0)
        period_key  = period_end.to_period("M").strftime("%Y-%m")

        if period_key in cached_months:
            continue  # already parsed

        print(f"  [Parsing] {fname} ...", end=" ")
        values = _parse_single_pdf(pdf_path)
        if values is None:
            continue

        # Convert IDR millions → IDR billions for consistency
        row = {"ds": period_end}
        for col, val in values.items():
            row[col] = val / 1_000 if not np.isnan(val) else np.nan
        new_rows.append(row)
        print(f"NI={row.get('net_profit', '?'):.0f}B  "
              f"Rev={row.get('interest_income', '?'):.0f}B")

    # ── 3. Merge new rows with cache ────────────────────────────────────
    if new_rows:
        new_df = pd.DataFrame(new_rows)
        all_raw = pd.concat([cached, new_df], ignore_index=True) if not cached.empty else new_df
        all_raw = all_raw.sort_values("ds").reset_index(drop=True)
        all_raw.to_csv(cache_path, index=False)
        print(f"[Monthly]  {len(new_rows)} new months parsed and saved to cache.")
    else:
        all_raw = cached.copy()

    # ── 4. Rename raw columns to model-facing names ──────────────────────
    raw = all_raw.rename(columns={
        "net_profit":       "net_income",   # align with old column names
        "interest_income":  "revenue",
    }).sort_values("ds").reset_index(drop=True)

    # ── 5. Compute derived features ──────────────────────────────────────

    # Net profit margin
    raw["net_profit_margin"] = raw["net_income"] / raw["revenue"]

    # Loan-to-asset ratio (balance-sheet measure of credit exposure)
    if "total_loans" in raw.columns and "total_assets" in raw.columns:
        raw["loans_to_assets"] = raw["total_loans"] / raw["total_assets"]
    else:
        raw["loans_to_assets"] = np.nan

    # NIM proxy: net interest income / total assets
    # Only computable if both interest_expense and total_assets are available
    if "interest_expense" in raw.columns and "total_assets" in raw.columns:
        net_interest = raw["revenue"] - raw["interest_expense"]
        raw["nim_proxy"] = net_interest / raw["total_assets"]

    # MoM growth rates
    raw["ni_growth_mom"]  = raw["net_income"].pct_change(1).fillna(0).clip(-1, 2)
    raw["rev_growth_mom"] = raw["revenue"].pct_change(1).fillna(0).clip(-1, 2)

    # YoY growth rates (12-month lag) — more meaningful than QoQ for financials
    raw["ni_growth_yoy"]  = raw["net_income"].pct_change(12).fillna(0).clip(-1, 2)
    raw["rev_growth_yoy"] = raw["revenue"].pct_change(12).fillna(0).clip(-1, 2)

    # ── 6. Select and report ─────────────────────────────────────────────
    keep = ["ds"] + FUNDAMENTAL_COLS
    # Add nim_proxy if available
    if "nim_proxy" in raw.columns:
        keep.append("nim_proxy")

    fund = raw[keep].dropna(subset=["net_income", "revenue"]).reset_index(drop=True)

    print(f"[Monthly]  {len(fund)} months of fundamentals  "
          f"({fund['ds'].min().date()} -> {fund['ds'].max().date()})")
    return fund

## Section 3 — Align Monthly Features to Daily (Forward-Fill)

The alignment logic is the same as before, but now the source cadence is
**monthly** rather than quarterly. Forward-filling is still applied so that
on any trading day the model only sees the most recently published report
(point-in-time correctness).

Monthly data also means the forward-fill "staleness" window is ≤ 31 days
instead of ≤ 92 days with quarterly data — a 3× improvement in signal freshness.

In [19]:
# ==============================================================================
#  SECTION 3 — ALIGN MONTHLY FEATURES TO DAILY (FORWARD-FILL)
# ==============================================================================

def align_to_daily(
    df_price : pd.DataFrame,
    fund_m   : pd.DataFrame,
    feat_cols: list[str],
) -> pd.DataFrame:
    """
    Forward-fill (LOCF) monthly fundamentals onto every calendar day,
    then inner-join to actual trading days.

    Point-in-time correctness: on any trading day t, the model only sees
    values from the most recently published monthly report (period_end ≤ t).
    BCA typically publishes the monthly statement ~4-6 weeks after period end,
    so if you want to be conservative you can shift the ds forward by 45 days:

        fund_m["ds"] = fund_m["ds"] + pd.DateOffset(days=45)

    This is commented out below — enable it for stricter no-lookahead.
    """
    # Optionally apply publication lag (uncomment to enable)
    # fund_m = fund_m.copy()
    # fund_m["ds"] = fund_m["ds"] + pd.DateOffset(days=45)

    calendar = pd.DataFrame({
        "ds": pd.date_range(df_price["ds"].min(), df_price["ds"].max(), freq="D")
    })
    daily = calendar.merge(fund_m[["ds"] + feat_cols], on="ds", how="left")
    daily[feat_cols] = daily[feat_cols].ffill()
    daily = daily.dropna(subset=["net_income"])  # drop pre-first-report dates

    merged = df_price.merge(daily[["ds"] + feat_cols], on="ds", how="inner")
    merged = merged.dropna(subset=feat_cols).reset_index(drop=True)

    print(f"\n[Dataset]  {len(merged):,} rows  "
          f"({merged['ds'].min().date()} -> {merged['ds'].max().date()})")
    return merged

## Section 4 — Normalisation

In [20]:
# ==============================================================================
#  SECTION 4 — Z-SCORE NORMALISATION
# ==============================================================================

def normalise(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Z-score normalise regressor columns using training-set statistics only
    (fit on all rows; for strict correctness, fit on train only and transform
    both train and test — implement if you want to avoid any look-ahead).
    """
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue
        mu, sigma = df[col].mean(), df[col].std()
        df[col] = (df[col] - mu) / (sigma + 1e-9)
    return df

## Section 5 — Plot: Monthly Fundamentals

In [21]:
# ==============================================================================
#  SECTION 5 — PLOT: MONTHLY FUNDAMENTALS
# ==============================================================================

def plot_fundamentals(fund: pd.DataFrame) -> None:
    avail = [c for c in ["net_income", "revenue", "net_profit_margin",
                          "loans_to_assets", "ni_growth_yoy", "rev_growth_yoy"]
             if c in fund.columns]
    ncols = 2
    nrows = (len(avail) + 1) // 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
    axes = axes.flatten()

    labels = {
        "net_income":        "Net Income (IDR B)",
        "revenue":           "Interest Income (IDR B)",
        "net_profit_margin": "Net Profit Margin",
        "loans_to_assets":   "Loans / Assets",
        "ni_growth_yoy":     "NI YoY Growth",
        "rev_growth_yoy":    "Revenue YoY Growth",
    }
    for ax, col in zip(axes, avail):
        ax.plot(fund["ds"], fund[col], lw=1.0, color="steelblue")
        ax.set_title(labels.get(col, col), fontsize=9)
        ax.set_xlabel("Date"); ax.grid(True, alpha=0.3)

    for ax in axes[len(avail):]:
        ax.set_visible(False)

    fig.suptitle("BBCA Monthly Fundamentals — from OJK PDFs",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig("fundamental_features_monthly.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("[Saved] fundamental_features_monthly.png")

## Section 6 — Build & Train NeuralProphet Models

In [22]:
# ==============================================================================
#  SECTION 6 — BUILD & TRAIN NEURALPROPHET
# ==============================================================================

def build_model(regressor_cols: list[str] | None = None) -> NeuralProphet:
    model = NeuralProphet(
        n_lags          = N_LAGS,
        n_forecasts     = N_FORECASTS,
        yearly_seasonality  = True,
        weekly_seasonality  = True,
        daily_seasonality   = False,
        epochs          = EPOCHS,
        learning_rate   = LEARNING_RATE,
        loss_func       = "MAE",
    )
    if regressor_cols:
        for col in regressor_cols:
            model.add_lagged_regressor(col)
    return model


def train_and_predict(
    model     : NeuralProphet,
    train_df  : pd.DataFrame,
    full_df   : pd.DataFrame,
    model_name: str,
) -> pd.DataFrame:
    print(f"\n{'='*64}\n  {model_name}\n  Train: {len(train_df)}  Full: {len(full_df)}\n{'='*64}")
    model.fit(train_df, freq="B")  # B = business day
    forecast = model.predict(full_df)
    return forecast

## Section 7 — Evaluation Metrics

In [23]:
# ==============================================================================
#  SECTION 7 — EVALUATION
# ==============================================================================

def evaluate(y_true: np.ndarray, y_pred: np.ndarray, name: str) -> dict:
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mean_price = y_true.mean()
    da   = np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred))) * 100
    print(f"\n  -- {name}")
    print(f"     MAE      : {mae:>10,.2f}  ({mae/mean_price*100:.2f}%)")
    print(f"     RMSE     : {rmse:>10,.2f}  ({rmse/mean_price*100:.2f}%)")
    print(f"     Dir.Acc. : {da:.2f}%")
    return {"MAE": mae, "RMSE": rmse,
            "MAE_pct": mae/mean_price*100,
            "RMSE_pct": rmse/mean_price*100,
            "Dir_Acc": da}

## Section 8 — Comparison Plot

In [24]:
# ==============================================================================
#  SECTION 8 — PLOT: MODEL COMPARISON
# ==============================================================================

COLORS = {
    "Baseline":     "steelblue",
    "Hybrid":       "darkorange",
}

def plot_comparison(
    df_price : pd.DataFrame,
    forecasts: dict,          # {name: forecast_df}
    metrics  : dict,
    test_size: int,
) -> None:
    fig = plt.figure(figsize=(16, 14))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3)

    # Panel A — full history
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(df_price["ds"], df_price["y"], lw=0.8, color="black", label="Actual")
    for name, fc in forecasts.items():
        fc_v = fc.dropna(subset=["yhat1"])
        ax1.plot(fc_v["ds"], fc_v["yhat1"], lw=0.7,
                 color=COLORS.get(name, "grey"), alpha=0.75, label=name)
    ax1.set_title("BBCA Close Price — Full History with Model Fits")
    ax1.set_xlabel("Date"); ax1.set_ylabel("Close (IDR)")
    ax1.legend(loc="upper left"); ax1.grid(True, alpha=0.3)

    # Panel B — test window zoom
    ax2   = fig.add_subplot(gs[1, :])
    cutoff = df_price["ds"].iloc[-test_size]
    mask   = df_price["ds"] >= cutoff
    ax2.plot(df_price.loc[mask, "ds"], df_price.loc[mask, "y"],
             lw=1.0, color="black", label="Actual")
    for name, fc in forecasts.items():
        tfc = fc.loc[fc["ds"] >= cutoff].dropna(subset=["yhat1"])
        ax2.plot(tfc["ds"], tfc["yhat1"], lw=0.9,
                 color=COLORS.get(name, "grey"), alpha=0.85, label=name)
    ax2.set_title(f"Zoomed — Last {test_size} Trading Days (Test Set)")
    ax2.set_xlabel("Date"); ax2.set_ylabel("Close (IDR)")
    ax2.legend(loc="upper left"); ax2.grid(True, alpha=0.3)

    # Panel C — MAE / RMSE bars
    ax3   = fig.add_subplot(gs[2, 0])
    names = list(metrics.keys())
    x     = np.arange(len(names)); w = 0.35
    b1 = ax3.bar(x - w/2, [metrics[n]["MAE_pct"]  for n in names], w,
                 color=[COLORS.get(n, "grey") for n in names], alpha=0.90, label="MAE (%)")
    b2 = ax3.bar(x + w/2, [metrics[n]["RMSE_pct"] for n in names], w,
                 color=[COLORS.get(n, "grey") for n in names], alpha=0.55, label="RMSE (%)")
    ax3.bar_label(b1, fmt="%.2f%%", padding=2, fontsize=8)
    ax3.bar_label(b2, fmt="%.2f%%", padding=2, fontsize=8)
    ax3.set_title("Error Metrics (lower is better)")
    ax3.set_xticks(x); ax3.set_xticklabels(names, fontsize=9)
    ax3.set_ylabel("% of mean price"); ax3.legend(); ax3.grid(True, alpha=0.3, axis="y")

    # Panel D — directional accuracy
    ax4  = fig.add_subplot(gs[2, 1])
    bars = ax4.bar(names, [metrics[n]["Dir_Acc"] for n in names],
                   color=[COLORS.get(n, "grey") for n in names], alpha=0.88, width=0.4)
    ax4.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
    ax4.axhline(50, linestyle="--", color="grey", lw=0.8, label="Random (50%)")
    ax4.set_title("Directional Accuracy (higher is better)")
    ax4.set_ylabel("Accuracy (%)"); ax4.set_ylim(0, 100)
    ax4.legend(); ax4.grid(True, alpha=0.3, axis="y")

    plt.savefig("model_comparison_monthly.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("[Saved] model_comparison_monthly.png")

## Section 9 — Component Decomposition Plot

In [25]:
# ==============================================================================
#  SECTION 9 — PLOT: COMPONENT DECOMPOSITION
# ==============================================================================

def plot_components(forecast: pd.DataFrame, model_name: str = "Hybrid") -> None:
    print(f"\n[Components]  Plotting {model_name} decomposition...")
    fc = forecast.dropna(subset=["yhat1"]).copy()

    def _find(kws):
        for col in fc.columns:
            if all(k in col.lower() for k in kws):
                return col
        return None

    panels = []
    for title, kws, color in [
        ("Trend",                 ["trend"],          "steelblue"),
        ("Yearly Seasonality",    ["season", "year"], "steelblue"),
        ("Weekly Seasonality",    ["season", "week"], "steelblue"),
    ]:
        c = _find(kws)
        if c:
            panels.append((title, fc[c], color))

    ar_cols = [c for c in fc.columns if c.lower().startswith("ar")]
    if ar_cols:
        panels.append(("AR Component", fc[ar_cols].sum(axis=1), "mediumpurple"))

    for rc in [c for c in fc.columns if "lagged_regressor" in c.lower()]:
        nice = rc.replace("lagged_regressor_", "").replace("_", " ").title()
        panels.append((f"Regressor: {nice}", fc[rc], "darkorange"))

    if not panels:
        print("  [Warning] No decomposition columns found."); return

    fig, axes = plt.subplots(len(panels), 1,
                             figsize=(14, 3 * len(panels)), sharex=True)
    if len(panels) == 1:
        axes = [axes]

    fig.suptitle(
        f"{model_name} — Component Decomposition\n"
        "Trend | Seasonality | Autoregression | Monthly Fundamental Regressors",
        fontsize=11, fontweight="bold"
    )
    for ax, (title, series, color) in zip(axes, panels):
        ax.plot(fc["ds"], series, lw=0.9, color=color)
        ax.axhline(0, color="grey", lw=0.5, linestyle="--")
        ax.set_title(title, fontsize=9); ax.set_ylabel("Contribution", fontsize=8)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel("Date")
    plt.tight_layout()
    fname = f"components_{model_name.lower().replace(' ','_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Saved] {fname}")

## Section 10 — Summary Table & Main

In [26]:
# ==============================================================================
#  SECTION 10 — SUMMARY TABLE
# ==============================================================================

def print_summary(metrics: dict) -> None:
    names = list(metrics.keys())
    print("\n" + "=" * 72)
    print("  FINAL RESULTS — TWO-MODEL COMPARISON (Monthly Fundamentals)")
    print("=" * 72)
    print(f"  {'Metric':<26}" + "".join(f"{n:>15}" for n in names))
    print(f"  {'-'*68}")
    for label, key, fmt in [
        ("MAE (IDR)",     "MAE",      "{:>14,.2f}"),
        ("MAE (%)",       "MAE_pct",  "{:>13.2f}%"),
        ("RMSE (IDR)",    "RMSE",     "{:>14,.2f}"),
        ("RMSE (%)",      "RMSE_pct", "{:>13.2f}%"),
        ("Dir. Accuracy", "Dir_Acc",  "{:>13.2f}%"),
    ]:
        print(f"  {label:<26}" + "".join(fmt.format(metrics[n][key]) for n in names))
    print(f"  {'-'*68}")
    bl_mae, bl_rmse = metrics["Baseline"]["MAE"], metrics["Baseline"]["RMSE"]
    for n in names[1:]:
        dm = (bl_mae  - metrics[n]["MAE"])  / bl_mae  * 100
        dr = (bl_rmse - metrics[n]["RMSE"]) / bl_rmse * 100
        print(f"  {n} vs Baseline  ->  MAE {dm:+.2f}%   RMSE {dr:+.2f}%")
    print("=" * 72)


# ==============================================================================
#  MAIN
# ==============================================================================

def main():
    # ── 1. Load data ────────────────────────────────────────────────────
    df_price = load_price_data(PRICE_PATH)
    fund_m   = load_monthly_pdf_data(PDF_DIR, PARSED_CACHE)

    # Determine which regressor cols are actually present
    avail_cols = [c for c in FUNDAMENTAL_COLS if c in fund_m.columns]
    # Also include nim_proxy if it was computed
    if "nim_proxy" in fund_m.columns:
        avail_cols.append("nim_proxy")

    # ── 2. Diagnostic plot ───────────────────────────────────────────────
    plot_fundamentals(fund_m)

    # ── 3. Align to daily & normalise ───────────────────────────────────
    df_full = align_to_daily(df_price, fund_m, avail_cols)
    df_full = normalise(df_full, avail_cols)

    df_bl  = df_full[["ds", "y"]].copy()
    df_hyb = df_full[["ds", "y"] + avail_cols].copy()

    # ── 4. Train / test split ────────────────────────────────────────────
    train_bl  = df_bl.iloc[:-TEST_SIZE].copy()
    train_hyb = df_hyb.iloc[:-TEST_SIZE].copy()

    # ── 5. Build & train models ──────────────────────────────────────────
    fc_bl  = train_and_predict(
        build_model(),              train_bl,  df_bl,  "Baseline (Price-Only)")
    fc_hyb = train_and_predict(
        build_model(avail_cols),    train_hyb, df_hyb, "Hybrid (Price + Monthly Fundamentals)")

    # ── 6. Extract test-window predictions ───────────────────────────────
    cutoff = df_bl["ds"].iloc[-TEST_SIZE]

    def _tp(fc):
        return fc.loc[fc["ds"] >= cutoff, "yhat1"].values

    y_true  = df_bl.loc[df_bl["ds"] >= cutoff, "y"].values
    yp_bl   = _tp(fc_bl)
    yp_hyb  = _tp(fc_hyb)

    n = min(len(y_true), len(yp_bl), len(yp_hyb))
    y_true = y_true[-n:]; yp_bl = yp_bl[-n:]; yp_hyb = yp_hyb[-n:]

    # ── 7. Evaluate ───────────────────────────────────────────────────────
    print(f"\n{'='*64}\n  EVALUATION — last {TEST_SIZE} trading days\n{'='*64}")
    m_bl  = evaluate(y_true, yp_bl,  "Baseline")
    m_hyb = evaluate(y_true, yp_hyb, "Hybrid")
    all_m = {"Baseline": m_bl, "Hybrid": m_hyb}

    # ── 8. Plots ──────────────────────────────────────────────────────────
    plot_comparison(df_bl, {"Baseline": fc_bl, "Hybrid": fc_hyb}, all_m, TEST_SIZE)
    plot_components(fc_hyb, "Hybrid")

    # ── 9. Summary ────────────────────────────────────────────────────────
    print_summary(all_m)

## Run

In [27]:
main()

[Price]  2015-01-02 -> 2025-12-30  |  2,659 trading days
[Monthly]  Cache loaded: 106 months already parsed.
[Monthly]  106 months of fundamentals  (2015-03-31 -> 2025-11-30)
[Saved] fundamental_features_monthly.png

[Dataset]  2,598 rows  (2015-03-31 -> 2025-12-30)

  Baseline (Price-Only)
  Train: 2358  Full: 2598
Training: |          | 0/? [00:23<?, ?it/s, v_num=44, train_loss=0.0102, reg_loss=0.000, MAE=97.20, RMSE=137.0, Loss=0.0102, RegLoss=0.000]  
Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 166.38it/s]

  Hybrid (Price + Monthly Fundamentals)
  Train: 2358  Full: 2598
Epoch 100: 100%|██████████| 100/100 [00:00<00:00, 130.26it/s]  
Training: |          | 0/? [00:21<?, ?it/s, v_num=45, train_loss=0.0117, reg_loss=0.000, MAE=118.0, RMSE=200.0, Loss=0.0117, RegLoss=0.000]  
Predicting DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 149.87it/s]

  EVALUATION — last 240 trading days

  -- Baseline
     MAE      :     295.33  (3.44%)
     RMSE     :     393.07  (4.58%

---
## Appendix A — Debug: Inspect a Single PDF

Run this cell to check what text pdfplumber extracts from any PDF,
and whether the regex patterns fire correctly. Useful when a month
shows NaN for some fields.

In [28]:
# ── DEBUG: test one PDF ─────────────────────────────────────────────────────
# Change this path to any PDF you want to inspect
DEBUG_PDF = "monthly_reports/BBCA_2015_08.pdf"

import pdfplumber

with pdfplumber.open(DEBUG_PDF) as pdf:
    full_text = "\n".join(p.extract_text() or "" for p in pdf.pages)

print("=== RAW TEXT (first 3000 chars) ===")
print(full_text[:3000])

print("\n=== EXTRACTED VALUES ===")
values = _extract_from_text(full_text)
for k, v in values.items():
    print(f"  {k:25s}: {v:>15,.0f} M IDR  ({v/1000:>10,.1f} B IDR)" if not np.isnan(v) else f"  {k:25s}: NOT FOUND")

=== RAW TEXT (first 3000 chars) ===
PT BANK CENTRAL ASIA Tbk
STATEMENTS OF FINANCIAL POSITION
As of August 31, 2015
(In millions of Rupiah)
BANK
No. ACCOUNTS Unaudited
August 31, 2015
ASSETS
1. Cash 13,815,178
2. Placement to Bank Indonesia 68,082,675
3. Interbank placement 13,076,870
4. Spot and derivatives claims 26,646
5. Securities 67,292,481
a. Measured at fair value through profit and loss 1,137,387
b. Available for sale 49,059,656
c. Held to maturity 14,255,063
d. Loan and receivables 2,840,375
6. Securities sold under repurchase agreement
(repo) -
7. Claims on securities bought under reverse
repo 27,354,383
8. Acceptance claims 6,444,849
9. Loans 358,988,534
a. Measured at fair value through profit and loss -
b. Available for sale -
c. Held to maturity -
d. Loan and receivables 358,988,534
10. Sharia Financing -
11. Equity investment 1,848,475
12. Impairment on financial assets -/- (8,187,231)
a. Securities (737,810)
b. Loans (7,209,370)
c. Others (240,051)
13. Intangible asset

## Appendix B — Regex Tuning Guide

If the debug cell above shows `NOT FOUND` for some fields, the PDF layout
may differ from the patterns in `_PATTERNS`. Here's how to fix it:

1. Run the debug cell and find the relevant line in `RAW TEXT`.
2. Copy the exact label text as it appears (including any line breaks).
3. Add a new pattern tuple to `_PATTERNS` for that label variant.

Common causes of mismatch:
- **Line break in the middle of the label** — pdfplumber joins hyphenated words
  differently across PDF versions. Use `[\s\S]{0,30}?` to span short gaps.
- **Indonesian-only layout** — early reports (pre-2017) may be Indonesian only.
  Indonesian equivalents are already included for the main fields.
- **Image-based PDF** — if `RAW TEXT` is empty or shows garbled characters,
  the PDF is a scanned image. Use `pytesseract` OCR:
  ```
  pip install pytesseract Pillow
  # Then convert pages to images and run OCR before regex.
  ```